# 1D Unit Random Walk: Return-to-Origin Simulation

This notebook simulates the classic 1D unit random walk:
- Start at x = 0
- Move +1 or -1 with equal probability each step
- Stop when the walk returns to x = 0 after leaving it

All files are saved in `outputs/unit_walk_1D/`.

In [ ]:
from pathlib import Path
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils import (
    ensure_output_dirs,
    plot_histogram,
    plot_overlay_1d,
    plot_single_walk_1d,
    save_1d_path_json,
    save_summary_csv,
    simulate_unit_walk_1d,
)

# Configuration
RANDOM_SEED = 20260407
N_TRIALS = 300
MAX_STEPS = 20_000

# Simple assumption: run this notebook from inside random_walks/
NOTEBOOK_DIR = Path('.')
OUTPUT_DIR, PATHS_DIR = ensure_output_dirs(NOTEBOOK_DIR / 'outputs' / 'unit_walk_1D')

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(RANDOM_SEED)

print(f'Working directory: {NOTEBOOK_DIR.resolve()}')
print(f'Output directory:  {OUTPUT_DIR.resolve()}')

## Mathematical Description

Let each step be xi_k in {+1, -1}, with probability 1/2 each.

Position after n steps is:
`X_n = sum_{k=1..n} xi_k`, with `X_0 = 0`.

We stop each trial at the first return time:
`T = min{ n >= 1 : X_n = 0 }`.

Since return times can be long, each run has a `MAX_STEPS` safeguard.
If no return is observed before that, the trial is marked as censored.

In [ ]:
def make_trial_record(walk: Dict[str, object], trial_index: int) -> Dict[str, object]:
    return {
        'trial_index': trial_index,
        'steps_until_stop': walk['steps_until_stop'],
        'return_time': walk['return_time'],
        'returned_to_origin': walk['returned_to_origin'],
        'censored': walk['censored'],
        'max_abs_displacement': walk['max_abs_displacement'],
        'min_position': walk['min_position'],
        'max_position': walk['max_position'],
        'final_position': walk['final_position'],
    }


def compute_empirical_metrics(summary_df: pd.DataFrame) -> Dict[str, float]:
    completed = summary_df[~summary_df['censored']]
    return {
        'completed_trials': float(len(completed)),
        'mean_return_time': float(completed['return_time'].mean()) if len(completed) else float('nan'),
        'variance_return_time': float(completed['return_time'].var(ddof=1)) if len(completed) > 1 else float('nan'),
        'mean_max_abs_displacement': float(summary_df['max_abs_displacement'].mean()),
    }

## Single Example Walk

Simulate one trajectory and visualize position versus step number.

In [ ]:
single_walk = simulate_unit_walk_1d(rng=rng, max_steps=MAX_STEPS)

single_walk_plot_path = OUTPUT_DIR / 'single_walk.png'
plot_single_walk_1d(
    positions=single_walk['positions'],
    out_path=single_walk_plot_path,
    title='Single 1D Unit Walk (Position vs Step)',
)

print('Single walk summary:')
print(f"  steps_until_stop:      {single_walk['steps_until_stop']}")
print(f"  returned_to_origin:    {single_walk['returned_to_origin']}")
print(f"  max_abs_displacement:  {single_walk['max_abs_displacement']}")
print(f'  saved plot:            {single_walk_plot_path}')

## Monte Carlo Simulation

Run many independent trials. For each trial we save:
- summary metrics in `summary.csv`
- raw path data in `outputs/unit_walk_1D/paths/`

In [ ]:
trial_records: List[Dict[str, object]] = []
all_paths: List[List[int]] = []

for trial_idx in range(N_TRIALS):
    walk = simulate_unit_walk_1d(rng=rng, max_steps=MAX_STEPS)
    all_paths.append(walk['positions'])
    save_1d_path_json(walk['positions'], trial_idx, PATHS_DIR)
    trial_records.append(make_trial_record(walk, trial_idx))

summary_df = pd.DataFrame(trial_records)
summary_csv_path = save_summary_csv(summary_df, OUTPUT_DIR / 'summary.csv')

metrics = compute_empirical_metrics(summary_df)

print(f'Trials run:                      {N_TRIALS}')
print(f"Completed returns:               {int(metrics['completed_trials'])}")
print(f"Censored trials:                 {int(summary_df['censored'].sum())}")
print(f"Empirical mean return time:      {metrics['mean_return_time']:.3f}")
print(f"Empirical variance return time:  {metrics['variance_return_time']:.3f}")
print(f"Empirical mean max |x|:          {metrics['mean_max_abs_displacement']:.3f}")
print(f'Saved summary CSV:               {summary_csv_path}')

summary_df.head()

## Visualization of Monte Carlo Results

Plot histograms for return times and max absolute displacement.

In [ ]:
completed_return_times = summary_df.loc[~summary_df['censored'], 'return_time']

return_hist_path = OUTPUT_DIR / 'return_time_histogram.png'
max_dist_hist_path = OUTPUT_DIR / 'max_distance_histogram.png'

plot_histogram(
    values=completed_return_times,
    out_path=return_hist_path,
    title='Return Time Histogram (Completed Trials)',
    xlabel='Steps until first return',
    color='tab:blue',
)

plot_histogram(
    values=summary_df['max_abs_displacement'],
    out_path=max_dist_hist_path,
    title='Max Absolute Displacement Histogram',
    xlabel='Max |position| during walk',
    color='tab:orange',
)

print(f'Saved: {return_hist_path}')
print(f'Saved: {max_dist_hist_path}')

## Final Overlay Plot

Overlay all position-vs-step trajectories on one figure.

In [ ]:
overlay_path = OUTPUT_DIR / 'overlay_walks.png'
plot_overlay_1d(
    all_paths=all_paths,
    out_path=overlay_path,
    title='Overlay of All 1D Walks (Position vs Step)',
)
print(f'Saved: {overlay_path}')